# 🦷 Khám Phá & Phân Tích Dữ Liệu (EDA) - CODeD Dataset

Notebook này phục vụ mục đích **tải, lưu trữ cục bộ và phân tích trực quan hóa dữ liệu (Exploratory Data Analysis - EDA)** cho bộ dữ liệu nha khoa đa phương thức **CODeD (Casangels Oro-dental Dataset)**.

### 📌 Nội dung chính:
1. **Cài đặt thư viện** xử lý dữ liệu và trực quan hóa (`pandas`, `seaborn`, `matplotlib`, `Pillow`,...)
2. **Tải dataset** đầy đủ từ Hugging Face (`zirak-ai/COde`) và giải nén vào thư mục local `./data`
3. **Thống kê tổng quan** các tệp dữ liệu bảng, ảnh chụp, và nhãn JSON
4. **Phân tích nhân khẩu học**: Độ tuổi, giới tính của bệnh nhân
5. **Phân tích số lượng hình ảnh**: Ảnh chụp khoang miệng (Photographs) và phim X-quang (Radiographs)
6. **Trực quan hóa hình ảnh thực tế**: Xem một số ảnh chụp khoang miệng và X-quang kèm thông tin ca bệnh
7. **Phân tích bệnh lý (Diseases & Diagnosis)**: Tần suất các loại bệnh lý phổ biến nhất
8. **Phân tích độ dài câu hỏi/trả lời** trong các tập dữ liệu hỏi đáp (QA Benchmark)

In [ ]:
# 1. Cài đặt các thư viện cần thiết cho EDA
!pip install -q -U huggingface_hub pandas matplotlib seaborn pillow numpy

In [ ]:
# 2. Import các thư viện và cấu hình hiển thị biểu đồ
import os
import json
import zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Thiết lập giao diện biểu đồ hiện đại
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'sans-serif']
plt.rcParams['figure.dpi'] = 120
print("Đã nạp xong các thư viện cần thiết!")

--- 
## 📥 Bước 1: Tải & Giải Nén Dataset Cục Bộ

In [ ]:
import os
# 3. Tải dataset từ Hugging Face về thư mục local ./COde_HF
import os
from huggingface_hub import snapshot_download
from pathlib import Path

# Cấu hình token Hugging Face để tăng tốc độ tải và bỏ qua giới hạn unauthenticated
HF_TOKEN = os.environ.get("HF_TOKEN", "")
os.environ["HF_TOKEN"] = HF_TOKEN

SAVE_DIR = Path("./COde_HF")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Nếu đã có file zip rồi thì không cần tải lại
zip_exists = any(SAVE_DIR.rglob("COde-Dataset.zip"))
if not zip_exists:
    print(f"Đang tải dataset zirak-ai/COde về {SAVE_DIR.resolve()} với HF_TOKEN...")
    snapshot_download(
        repo_id="zirak-ai/COde",
        repo_type="dataset",
        local_dir=SAVE_DIR,
        token=HF_TOKEN
    )
    print("Tải hoàn tất!")
else:
    print("Dataset đã tồn tại trong thư mục local, bỏ qua bước tải.")

In [ ]:
# 4. Giải nén dữ liệu vào ./data (nếu chưa giải nén)
ZIP_PATH = next(SAVE_DIR.rglob("COde-Dataset.zip"))
EXTRACT_DIR = Path("./data")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

complete_csv = EXTRACT_DIR / "complete_dataset.csv"
if not complete_csv.exists():
    print(f"Đang giải nén {ZIP_PATH.name} vào {EXTRACT_DIR.resolve()}...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
    print("Giải nén hoàn tất!")
else:
    print(f"Dữ liệu đã được giải nén sẵn tại {EXTRACT_DIR.resolve()}")

--- 
## 📊 Bước 2: Thống Kê Tổng Quan Cấu Trúc Dữ Liệu

In [ ]:
# 5. Kiểm kê các file và ảnh trong dataset
photos = list(EXTRACT_DIR.rglob("Images/Photographs/*"))
xrays = list(EXTRACT_DIR.rglob("Images/Radiographs/*"))
jsons = list(EXTRACT_DIR.glob("*.json"))
csvs = list(EXTRACT_DIR.glob("*.csv"))

print("========== TỔNG QUAN TÀI NGUYÊN ==========")
print(f"📷 Ảnh chụp khoang miệng (Photographs): {len(photos):,} ảnh")
print(f"🩻 Phim chụp X-quang (Radiographs)    : {len(xrays):,} ảnh")
print(f"🖼️ Tổng số hình ảnh đa phương thức     : {len(photos) + len(xrays):,} ảnh")
print(f"📄 Số tệp JSON chú thích              : {len(jsons)} tệp")
print(f"📊 Số tệp bảng CSV                    : {len(csvs)} tệp")
print("==========================================")

# Biểu đồ cột thể hiện phân bổ loại hình ảnh
plt.figure(figsize=(6, 4))
img_counts = pd.Series({"Photographs (Ảnh miệng)": len(photos), "Radiographs (Phim X-quang)": len(xrays)})
ax = sns.barplot(x=img_counts.index, y=img_counts.values, palette="Blues_d")
for i, v in enumerate(img_counts.values):
    ax.text(i, v + 500, f"{v:,}", ha="center", fontweight="bold", fontsize=11)
plt.title("Phân Bổ Loại Hình Ảnh Đa Phương Thức Trong CODeD", fontsize=13, pad=12)
plt.ylabel("Số Lượng Ảnh")
plt.ylim(0, max(img_counts.values) * 1.15)
plt.show()

--- 
## 👥 Bước 3: Phân Tích Dữ Liệu Lâm Sàng & Nhân Khẩu Học (Demographics)

In [ ]:
# 6. Đọc tệp dữ liệu hồ sơ lâm sàng complete_dataset.csv
df = pd.read_csv(EXTRACT_DIR / "complete_dataset.csv")
print(f"Kích thước tập hồ sơ lâm sàng: {df.shape[0]} dòng, {df.shape[1]} cột")
display(df.head(3))

# Kiểm tra các cột dữ liệu và tỷ lệ giá trị thiếu (Missing values)
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
plt.figure(figsize=(10, 4))
sns.barplot(x=missing_pct.index, y=missing_pct.values, color="#4C72B0")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Tỷ lệ còn thiếu (%)")
plt.title("Tỷ Lệ Thiếu Dữ Liệu (Missing Values) Theo Từng Thuộc Tính", fontsize=12, pad=10)
plt.show()

In [ ]:
# 7. Trực quan hóa phân bố Độ tuổi và Giới tính
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 7.1 Phân bố độ tuổi
age_clean = df["age"].dropna()
sns.histplot(age_clean, kde=True, bins=30, ax=axes[0], color="teal")
axes[0].set_title(f"Phân Bố Độ Tuổi Bệnh Nhân (Mean: {age_clean.mean():.1f}, Median: {age_clean.median():.0f})")
axes[0].set_xlabel("Tuổi")
axes[0].set_ylabel("Số Lượng Ca Khám")

# 7.2 Phân bố giới tính
gender_counts = df["gender"].fillna("Không rõ").value_counts()
axes[1].pie(
    gender_counts,
    labels=gender_counts.index,
    autopct="%1.1f%%",
    colors=["#4878D0", "#EE854A", "#6ACC64"],
    startangle=140
)
axes[1].set_title("Tỷ Lệ Giới Tính Bệnh Nhân")

# 7.3 Phân bố độ tuổi theo giới tính
df_valid_demo = df.dropna(subset=["age", "gender"])
df_valid_demo = df_valid_demo[df_valid_demo["gender"].isin(["Male", "Female", "男", "女"])]
# Chuẩn hóa tên giới tính hiển thị
gender_map = {"男": "Male", "女": "Female", "Male": "Male", "Female": "Female"}
df_valid_demo["gender_norm"] = df_valid_demo["gender"].map(gender_map)

sns.boxplot(data=df_valid_demo, x="gender_norm", y="age", ax=axes[2], palette="Set2")
axes[2].set_title("Phân Phối Độ Tuổi Theo Giới Tính")
axes[2].set_xlabel("Giới Tính")
axes[2].set_ylabel("Tuổi")

plt.tight_layout()
plt.show()

--- 
## 📷 Bước 4: Phân Tích Số Lượng Ảnh Trên Mỗi Ca Khám & Xem Mẫu Ảnh

In [ ]:
# 8. Phân tích số lượng ảnh mỗi lần khám (Checkup)
df["num_photos"] = df["photographs"].fillna("").apply(lambda x: len(str(x).split(",")) if str(x).strip() else 0)
df["num_xrays"] = df["radiographs"].fillna("").apply(lambda x: len(str(x).split(",")) if str(x).strip() else 0)
df["total_images"] = df["num_photos"] + df["num_xrays"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

sns.countplot(data=df[df["num_photos"] <= 15], x="num_photos", ax=axes[0], color="#3B528B")
axes[0].set_title("Phân Bố Số Ảnh Chụp Miệng (Photographs) Mỗi Ca")
axes[0].set_xlabel("Số Lượng Ảnh Chụp")
axes[0].set_ylabel("Số Ca Khám")

sns.countplot(data=df[df["num_xrays"] <= 8], x="num_xrays", ax=axes[1], color="#5DC863")
axes[1].set_title("Phân Bố Số Phim X-quang (Radiographs) Mỗi Ca")
axes[1].set_xlabel("Số Lượng Phim X-quang")
axes[1].set_ylabel("Số Ca Khám")

plt.tight_layout()
plt.show()

In [ ]:
# 9. Hiển thị một số hình ảnh thực tế từ dataset kèm chẩn đoán
sample_rows = df[df["num_photos"] > 0].sample(n=min(3, len(df)), random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, (_, row) in enumerate(sample_rows.iterrows()):
    first_photo = str(row["photographs"]).split(",")[0].strip()
    img_path = EXTRACT_DIR / "Images" / "Photographs" / first_photo
    
    if img_path.exists():
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].axis("off")
        diag_text = str(row.get('diagnosis', 'N/A'))[:60] + '...'
        axes[idx].set_title(f"Ca: {row['checkup_id']} | Tuổi: {row['age']}\n{diag_text}", fontsize=10)
    else:
        axes[idx].text(0.5, 0.5, f"Không tìm thấy:\n{first_photo}", ha="center", va="center")
        axes[idx].axis("off")

plt.suptitle("Một Số Hình Ảnh Lâm Sàng Thực Tế Trong Bộ Dữ Liệu", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

--- 
## 🩺 Bước 5: Phân Tích Các Loại Bệnh Lý (Disease Distribution)

In [ ]:
# 10. Phân tích phân bố bệnh lý từ tập phân loại test_cls.json và train-cls.json
cls_file = EXTRACT_DIR / "test_cls.json"
if not cls_file.exists():
    cls_file = EXTRACT_DIR / "train-cls.json"

if cls_file.exists():
    with open(cls_file, "r", encoding="utf-8") as f:
        cls_data = json.load(f)
    
    # Trích xuất nhãn câu trả lời (tên bệnh lý)
    diseases = [sample["messages"][1]["content"].strip() for sample in cls_data if len(sample.get("messages", [])) > 1]
    disease_counts = pd.Series(diseases).value_counts().head(15)
    
    plt.figure(figsize=(12, 6))
    bars = sns.barplot(x=disease_counts.values, y=disease_counts.index, palette="mako")
    for i, v in enumerate(disease_counts.values):
        plt.text(v + 5, i, f"{v:,}", va="center", fontweight="bold", fontsize=10)
    
    plt.title(f"Top {len(disease_counts)} Bệnh Lý Răng Miệng Phổ Biến Nhất ({cls_file.name})", fontsize=13, pad=12)
    plt.xlabel("Số Lượng Mẫu")
    plt.ylabel("Tên Bệnh Lý")
    plt.xlim(0, max(disease_counts.values) * 1.15)
    plt.show()
else:
    print("Chưa tìm thấy file test_cls.json hoặc train-cls.json để phân tích nhãn bệnh.")

--- 
## 📝 Bước 6: Phân Tích Đặc Trưng Prompt & Ngữ Cảnh QA

In [ ]:
# 11. Phân tích độ dài câu hỏi và câu trả lời trong tập Benchmark
benchmarks = {}
for fname in ["test_cls.json", "test_diagnostic.json"]:
    fpath = EXTRACT_DIR / fname
    if fpath.exists():
        with open(fpath, "r", encoding="utf-8") as f:
            benchmarks[fname] = json.load(f)

if benchmarks:
    stats = []
    for bench_name, samples in benchmarks.items():
        for s in samples:
            msgs = s.get("messages", [])
            if len(msgs) >= 2:
                stats.append({
                    "Benchmark": bench_name,
                    "User_Prompt_Chars": len(msgs[0]["content"]),
                    "User_Prompt_Words": len(msgs[0]["content"].split()),
                    "Assistant_Answer_Chars": len(msgs[1]["content"]),
                    "Assistant_Answer_Words": len(msgs[1]["content"].split()),
                    "Num_Images": len(s.get("images", []))
                })
    
    df_stats = pd.DataFrame(stats)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    
    # Độ dài câu trả lời của mô hình
    sns.boxplot(data=df_stats, x="Benchmark", y="Assistant_Answer_Words", ax=axes[0], palette="Set2")
    axes[0].set_title("Phân Bố Độ Dài Câu Trả Lời (Số Từ - Words)")
    axes[0].set_ylabel("Số Từ (Words)")
    
    # Phân bố số ảnh đính kèm trên mỗi câu hỏi
    sns.countplot(data=df_stats, x="Num_Images", hue="Benchmark", ax=axes[1], palette="Set1")
    axes[1].set_title("Số Lượng Ảnh Đính Kèm Mỗi Lượt Hỏi")
    axes[1].set_xlabel("Số Ảnh")
    axes[1].set_ylabel("Số Lượng Mẫu")
    
    plt.tight_layout()
    plt.show()
    
    print("\n--- TỔNG KẾT ĐẶC TRƯNG TẬP BENCHMARK ---")
    display(df_stats.groupby("Benchmark").describe()[["Num_Images", "Assistant_Answer_Words"]])

--- 
## 🏁 Kết Luận EDA

Qua các bước khám phá trên, chúng ta đã nắm được:
- **Quy mô đa phương thức**: Hơn 57,000 ảnh (bao gồm cả ảnh chụp khoang miệng thực tế và phim X-quang) phân bổ đều cho từng ca khám.
- **Nhân khẩu học**: Dữ liệu trải dài nhiều độ tuổi từ thanh thiếu niên đến người cao tuổi, cân bằng giới tính.
- **Đa dạng tác vụ**: Bộ dữ liệu được chia làm 2 nhánh rõ rệt:
  1. **Phân loại bệnh lý (Classification)**: Trả về tên bệnh chuẩn hóa ngắn gọn.
  2. **Báo cáo chẩn đoán (Diagnostic Generation)**: Trả về báo cáo lâm sàng toàn diện dài gồm khám, chẩn đoán, và kế hoạch điều trị.